In [ ]:
from xgboost import XGBClassifier
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.model_selection import GridSearchCV

In [ ]:
df = pd.read_csv("../../../data/processed/feature_dataset.csv")

In [ ]:
# One hot encoding for age and recording location
df = pd.get_dummies(df, columns=["Age"], drop_first=True)

df = pd.get_dummies(df, columns=["recording_location"], drop_first=True)

df = pd.get_dummies(df, columns=["Murmur"], drop_first=True)

df["Sex"] = df["Sex"].map({
    "Female": 0,
    "Male": 1
})

df["Outcome"] = df["Outcome"].map({
    "Normal": 0,
    "Abnormal": 1
})

# print(df.columns.tolist())

In [ ]:
# -----------------------------
# Split using your predefined split
# -----------------------------
train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

In [ ]:
# -----------------------------
# Prepare features
# -----------------------------
drop_cols = ["Patient ID", "Outcome", "split", "file", "Campaign", "Additional ID", "Height", "Weight"]

drop_cols = drop_cols + ["rms"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["Outcome"]

X_val = val_df.drop(columns=drop_cols)
y_val = val_df["Outcome"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["Outcome"]

In [ ]:
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import precision_score, recall_score
from xgboost import XGBClassifier
import numpy as np


param_grid = {
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.2],
    "scale_pos_weight": [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4]
}


# Minimum acceptable precision
min_precision = 0.70

best_model = None
best_params = None
best_threshold = None
best_recall = 0
best_precision = 0


for params in ParameterGrid(param_grid):

    print("Testing:", params)

    model = XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    )

    # Train
    model.fit(
        X_train,
        y_train
    )

    # Validation probabilities
    y_prob = model.predict_proba(X_val)[:, 1]


    # Try different thresholds
    for threshold in np.arange(0.05, 0.95, 0.01):

        y_pred = (y_prob >= threshold).astype(int)

        precision = precision_score(
            y_val,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_val,
            y_pred
        )


        # Maximize recall while keeping precision constraint
        if precision >= min_precision and recall > best_recall:

            best_recall = recall
            best_precision = precision
            best_threshold = threshold
            best_model = model
            best_params = params


print("\nBest parameters:")
print(best_params)

print("\nBest threshold:")
print(best_threshold)

print("\nValidation performance:")
print("Recall:", best_recall)
print("Precision:", best_precision)

In [ ]:
best_xgb_model = XGBClassifier(
    learning_rate=0.02,
    max_depth=5,
    scale_pos_weight=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

best_xgb_model.fit(
    X_train,
    y_train
)

In [ ]:
threshold = 0.45

# Test probabilities
y_prob = best_xgb_model.predict_proba(X_test)[:, 1]

# Custom threshold
y_pred = (y_prob >= threshold).astype(int)

In [ ]:
import pandas as pd

importance = (
    pd.Series(
        best_xgb_model.feature_importances_,
        index=X_train.columns
    )
    .sort_values(ascending=False)
)

print(importance.head(20))

In [ ]:
# EVALUATION
# ONLY RUN ONCE

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print(confusion_matrix(y_test, y_pred))